In [24]:
# Install dependencies
!pip install -q \
  langchain-core \
  langchain-community \
  langchain-text-splitters \
  langchain-huggingface \
  sentence-transformers \
  faiss-cpu


In [25]:
# Imports
import json
from google.colab import drive
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS


In [26]:
# Mount Google Drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [27]:
# Load your JSON from Drive
# Update this path if your file is inside a folder in MyDrive
json_path = "/content/drive/MyDrive/portfolio.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Loaded keys:", data.keys())


Loaded keys: dict_keys(['about', 'education', 'skills', 'experience', 'projects'])


In [28]:
# Convert JSON into text documents
documents = []

# About
documents.append(f"Section: About\n{data['about']}")

# Skills
skills_text = []
for category, items in data["skills"].items():
    pretty_category = category.replace("_", " ").title()
    skills_text.append(f"{pretty_category}: {', '.join(items)}")

documents.append("Section: Skills\n" + "\n".join(skills_text))

# Projects
for proj in data["projects"]:
    text = f"""
Section: Project
Name: {proj['name']}
Problem: {proj['problem']}
Solution: {proj['solution']}
Tech: {', '.join(proj['tech'])}
Impact: {proj['impact']}
Details: {proj['details']}
""".strip()
    documents.append(text)

# Experience
for exp in data["experience"]:
    text = f"""
Section: Experience
Role: {exp['role']}
Company: {exp['company']}
Timeline: {exp['timeline']}
Responsibilities: {'; '.join(exp['responsibilities'])}
Impact: {exp['impact']}
""".strip()
    documents.append(text)

print("Document count:", len(documents))
print(documents[0][:300])


Document count: 13
Section: About
Software engineer with a focus on backend systems, data engineering, and machine learning applications. Experienced in building scalable pipelines, automation systems, and data-driven solutions. Strong background in applied machine learning, distributed systems, and real-world problem


In [29]:
# Split into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = splitter.split_text("\n\n".join(documents))

print("Chunk count:", len(chunks))
print(chunks[0][:300])


Chunk count: 18
Section: About
Software engineer with a focus on backend systems, data engineering, and machine learning applications. Experienced in building scalable pipelines, automation systems, and data-driven solutions. Strong background in applied machine learning, distributed systems, and real-world problem


In [30]:
# Wrap chunks as LangChain Documents
docs = [
    Document(page_content=chunk, metadata={"source": "portfolio"})
    for chunk in chunks
]

print("Docs ready:", len(docs))


Docs ready: 18


In [31]:
# Build FAISS vector store
db = FAISS.from_documents(docs, embeddings)

print("FAISS index created")


FAISS index created


In [32]:
# Save vector store
save_path = "/content/drive/MyDrive/vectorstore_portfolio"
db.save_local(save_path)

print("Saved to:", save_path)


Saved to: /content/drive/MyDrive/vectorstore_portfolio


In [33]:
# Quick retrieval test
query = "What backend and machine learning experience does this person have?"
results = db.similarity_search(query, k=3)

for i, doc in enumerate(results, 1):
    print(f"\nResult {i}\n")
    print(doc.page_content)



Result 1

Section: About
Software engineer with a focus on backend systems, data engineering, and machine learning applications. Experienced in building scalable pipelines, automation systems, and data-driven solutions. Strong background in applied machine learning, distributed systems, and real-world problem solving through end-to-end project development.

Result 2

Section: Experience
Role: Machine Learning Intern
Company: Indian Servers
Timeline: May 2020 - Aug 2020
Responsibilities: Built a traffic sign recognition system using machine learning; Performed data preprocessing and feature engineering; Trained and evaluated classification models
Impact: Improved model accuracy and contributed to real-time prediction capabilities

Result 3

Section: Skills
Programming Languages: Python, C, SQL
Data Engineering: Apache Spark, Data Pipelines, Data Cleaning, Feature Engineering
Machine Learning: Scikit-Learn, TensorFlow, PyTorch, OpenCV, Deep Learning, Computer Vision
Backend And Tools: F